In [0]:
PAYMENTS_SOURCE_PATH = (
    "/Volumes/credlake/landing/raw/payments/"
)

PAYMENTS_SCHEMA_LOCATION = (
    "/Volumes/credlake/ops/pipeline_state/"
    "schemas/payments/"
)

PAYMENTS_CHECKPOINT_LOCATION = (
    "/Volumes/credlake/ops/pipeline_state/"
    "checkpoints/payments_bronze/"
)

TARGET_TABLE = "credlake.bronze.payments_raw"

In [0]:
from pyspark.sql import functions as F

payments_raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option(
        "cloudFiles.schemaLocation",
        PAYMENTS_SCHEMA_LOCATION
    )
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.partitionColumns", "event_date")
    .load(PAYMENTS_SOURCE_PATH)
)

In [0]:
payments_raw_stream.printSchema()

In [0]:
payments_bronze_stream = (
    payments_raw_stream
    .select(
        "*",
        F.col("_metadata.file_path")
            .alias("source_file_path"),
        F.col("_metadata.file_name")
            .alias("source_file_name"),
        F.col("_metadata.file_size")
            .alias("source_file_size"),
        F.col("_metadata.file_modification_time")
            .alias("source_file_modification_time")
    )
    .withColumn(
        "event_date",
        F.to_date(F.col("event_date"))
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
)

In [0]:
payments_bronze_stream.printSchema()

In [0]:
payments_query = (
    payments_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        PAYMENTS_CHECKPOINT_LOCATION
    )
    .option("mergeSchema", "true")
    .queryName("credlake_payments_bronze_autoloader")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

payments_query.awaitTermination()

print("Ingestão incremental de pagamentos concluída.")

In [0]:
if payments_query.lastProgress:
    print(payments_query.lastProgress)
else:
    print("A execução terminou sem progresso registrado.")

In [0]:
print(f"Streams ativos: {len(spark.streams.active)}")

In [0]:
spark.sql("""
COMMENT ON TABLE credlake.bronze.payments_raw IS
'Eventos de pagamento ingeridos incrementalmente com Auto Loader.'
""")

spark.sql("""
ALTER TABLE credlake.bronze.payments_raw
SET TBLPROPERTIES (
    'data_layer' = 'bronze',
    'data_domain' = 'payments',
    'ingestion_pattern' = 'auto_loader_available_now',
    'contains_synthetic_data' = 'true'
)
""")

In [0]:
spark.sql("""
DESCRIBE TABLE credlake.bronze.payments_raw
""").show(truncate=False)

In [0]:
display(
    spark.table("credlake.bronze.payments_raw")
    .limit(20)
)

In [0]:
payments_df = spark.table(
    "credlake.bronze.payments_raw"
)

required_columns = {
    "payment_id",
    "amount_paid",
    "source_file_path",
    "_rescued_data",
    "ingested_at"
}

missing_columns = required_columns - set(payments_df.columns)

assert not missing_columns, (
    f"Colunas obrigatórias ausentes: {missing_columns}"
)

payment_metrics_df = payments_df.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct("payment_id")
        .alias("distinct_payment_ids"),

    F.sum(
        F.when(
            F.col("payment_id").isNull(),
            1
        ).otherwise(0)
    ).alias("null_payment_ids"),

    F.sum(
        F.when(
            F.col("amount_paid").isNull(),
            1
        ).otherwise(0)
    ).alias("null_amount_paids"),

    F.sum(
        F.when(
            F.col("_rescued_data").isNotNull(),
            1
        ).otherwise(0)
    ).alias("rescued_records"),

    F.countDistinct("source_file_path")
        .alias("source_files")
)

display(payment_metrics_df)

In [0]:
metrics = payment_metrics_df.first()

duplicate_payment_rows = (
    metrics["total_rows"]
    - metrics["distinct_payment_ids"]
)

assert duplicate_payment_rows == 1, (
    "Era esperado exatamente um pagamento duplicado"
)

assert metrics["null_payment_ids"] == 0, (
    "Não eram esperados payment_id nulos"
)

assert metrics["null_amount_paids"] == 1, (
    "Era esperado exatamente um pagamento com valor nulo"
)

assert metrics["rescued_records"] == 0, (
    "Foram encontrados campos incompatíveis em _rescued_data"
)

print("Validações da Bronze de pagamentos concluídas.")